# figures


In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import paths as P

P.FIG_DIR.mkdir(parents=True, exist_ok=True)

URBAN_CORE = ["DNK.1.25_1", "DNK.2.1_1", "DNK.5.16_1", "DNK.3.1_1"]
NORDDJURS = "DNK.2.9_1"
DROP = {"DNK.1.4_1"}
MAP_DROP = {"DNK.1.4_1", "DNK.1.6_1"}
CUT = 25.1
ZSTAR = 28.0
PAD = 2.0
MIN_COMM_SIZE = 20
MIN_EDGES = 50
ALPHAS = np.logspace(-2, 3, 30)

COMM = {
    4: "Alcohol (wine, beer & spirits)",
    1: "Chilled deli, cheese & bread",
    0: "Specialty pantry & produce staples",
    2: "Sweet bakery, cereal & breakfast treats",
    17: "Candy, chocolate & salty snacks",
    3: "Dinner cooking (meat, pasta & sauces)",
    12: "Frozen pizza, meals & convenience meat",
    13: "Ice cream & dessert cupboard",
    8: "Ready meals & protein on-the-go",
    9: "Sausages & burger condiments",
    11: "Soft drinks, energy & functional beverages",
    14: "Jarred pesto, olives & antipasti",
}
SHORT = {
    "Alcohol (wine, beer & spirits)": "Wine & beer",
    "Chilled deli, cheese & bread": "Cheese & cold cuts",
    "Specialty pantry & produce staples": "Nuts & produce",
    "Sweet bakery, cereal & breakfast treats": "Breakfast & baked sweets",
    "Candy, chocolate & salty snacks": "Candy & snacks",
    "Dinner cooking (meat, pasta & sauces)": "Pasta & toppings",
    "Frozen pizza, meals & convenience meat": "Frozen meals",
    "Ice cream & dessert cupboard": "Ice cream & desserts",
    "Ready meals & protein on-the-go": "Ready meals",
    "Sausages & burger condiments": "Sausages & condiments",
    "Soft drinks, energy & functional beverages": "Soft drinks",
    "Jarred pesto, olives & antipasti": "Antipasti",
}
NAME_FIX = {
    "Aarhus": "Århus",
    "Høje-Taastrup": "HøjeTaastrup",
    "Vesthimmerlands": "Vesthimmerland",
}
RATE_MAP = {
    "rate_antidiabetics (A10)": "A10",
    "rate_cardiovascular (C00)": "C",
    "rate_cholesterol (C10)": "C10",
}
S_TOTAL = ["log_pop", "share_under30", "share_over60", "affluence_index"]
S_60 = ["log_pop", "affluence_index"]


In [ ]:
STORY = [
    "Fresh fruit", "Fresh vegetables", "Fish", "Dark bread", "White bread",
    "Low-fat milk", "Whole milk", "Meat", "Cheese", "Drinks",
]
FADING = [
    ("Fresh vegetables", "Cheese"),
    ("Fresh vegetables", "Fish"),
    ("Fresh vegetables", "Low-fat milk"),
    ("Fresh fruit", "Low-fat milk"),
]
RISING = [
    ("Drinks", "Drinks"),
    ("White bread", "Meat"),
    ("Fresh fruit", "Fresh fruit"),
    ("Fresh fruit", "Fresh vegetables"),
]
CTRL = [("Meat", "Cheese")]


def collapse(c):
    if c in ("White meat", "Red meat", "Processed meat"):
        return "Meat"
    return c


def pair_key(a, b):
    if a not in STORY or b not in STORY:
        return None
    ia, ib = STORY.index(a), STORY.index(b)
    return (a, b) if ia <= ib else (b, a)


load


In [ ]:
def have(*ps):
    return all(Path(p).exists() for p in ps)

prod = pd.read_parquet(P.PRODUCTS)
if "brand_micro_cat" not in prod.columns:
    prod["brand_micro_cat"] = prod["brand"].fillna("").astype(str) + "_" + prod["level_3_micro"].fillna("").astype(str)
prod["brand_micro_cat"] = prod["brand_micro_cat"].astype(str)
if "nova" not in prod.columns:
    for c in ("foodprox_nova", "nova_group_estimated"):
        if c in prod.columns:
            prod["nova"] = prod[c]
            break
if "fscore" not in prod.columns:
    for c in ("FScore", "f_score"):
        if c in prod.columns:
            prod["fscore"] = prod[c]
            break
keep = set(prod["brand_micro_cat"])
nova_map = {k: int(v) for k, v in zip(prod["brand_micro_cat"], prod["nova"]) if pd.notna(v)}
fs_map = {k: float(v) for k, v in zip(prod["brand_micro_cat"], prod["fscore"]) if pd.notna(v)}
if "meta" in prod.columns:
    node_cat = dict(zip(prod["brand_micro_cat"].astype(str), prod["meta"].map(collapse).astype(str)))
else:
    node_cat = {str(k): "Other" for k in prod["brand_micro_cat"]}
    print("no meta column, pair shares will be empty")
print("products", len(keep), "nova", len(nova_map), "fscore", len(fs_map))

files = sorted(p for p in P.MUNICIPAL_EDGES_DIR.glob("*.parquet") if p.stem not in DROP)
print("municipal files", len(files))


def load_g(path, weighted=False):
    cols = ["product_i", "product_j"]
    df = pd.read_parquet(path)
    if weighted and "weight" in df.columns:
        cols = ["product_i", "product_j", "weight"]
        df = df[cols]
        df = df[df.product_i.isin(keep) & df.product_j.isin(keep)]
        df = df[df.product_i != df.product_j]
        g = nx.from_pandas_edgelist(df, "product_i", "product_j", "weight")
    else:
        df = df[["product_i", "product_j"]]
        df = df[df.product_i.isin(keep) & df.product_j.isin(keep)]
        g = nx.Graph()
        g.add_edges_from(zip(df.product_i.astype(str), df.product_j.astype(str)))
    return g


graphs = {}
for p in files:
    graphs[p.stem] = load_g(p, weighted=False)
print("graphs", len(graphs), "nodes example", list(graphs)[0], graphs[list(graphs)[0]].number_of_nodes())

geo = gpd.read_file(P.GEOJSON)
if geo.crs is not None:
    try:
        geo = geo.to_crs(25832)
    except Exception:
        pass
if "GID_2" in geo.columns:
    geo["kommune_id"] = geo["GID_2"].astype(str)
elif "kommune_id" not in geo.columns:
    geo["kommune_id"] = geo.index.astype(str)
print(geo.columns.tolist()[:12])


## fig 1


In [ ]:
def spec(g):
    h = g.subgraph([n for n, d in g.degree() if d > 0]).copy()
    if h.number_of_nodes() == 0:
        return np.array([])
    return np.sort(nx.normalized_laplacian_spectrum(h))


def spec_dist(a, b, pad=PAD):
    n = max(len(a), len(b))
    if n == 0:
        return 0.0
    aa = np.pad(a, (0, n - len(a)), constant_values=pad)
    bb = np.pad(b, (0, n - len(b)), constant_values=pad)
    return float(np.linalg.norm(aa - bb))


spectra = {}
for k, g in graphs.items():
    spectra[k] = spec(g)
    if len(spectra) % 20 == 0:
        print("spectra", len(spectra))
print("spectra done", len(spectra))

rows = []
for k in spectra:
    d = float(np.mean([spec_dist(spectra[k], spectra[u]) for u in URBAN_CORE if u in spectra and u != k]))
    if k in URBAN_CORE:
        grp = "urban"
    elif d < CUT:
        grp = "near"
    else:
        grp = "mid_far"
    rows.append({"kommune_id": k, "D": d, "group": grp})
axis = pd.DataFrame(rows)
print(axis.group.value_counts().to_string())
print(axis[axis.kommune_id.isin(URBAN_CORE + [NORDDJURS])])

def spec_on(g, nodes):
    h = nx.Graph()
    h.add_nodes_from(nodes)
    h.add_edges_from(g.subgraph(nodes).edges())
    return np.sort(nx.normalized_laplacian_spectrum(h))


shared_rows = []
for k, g in graphs.items():
    vals = []
    for u in URBAN_CORE:
        if u == k or u not in graphs:
            continue
        inter = set(g.nodes()) & set(graphs[u].nodes())
        if len(inter) < 20:
            continue
        sa = spec_on(g, inter)
        sb = spec_on(graphs[u], inter)
        vals.append(float(np.linalg.norm(sa - sb)))
    shared_rows.append({"kommune_id": k, "D_shared": float(np.mean(vals)) if vals else np.nan})
sh = axis.merge(pd.DataFrame(shared_rows), on="kommune_id")
print("shared-node D vs D", spearmanr(sh.D, sh.D_shared, nan_policy="omit"))


In [ ]:
gdf = geo.merge(axis, on="kommune_id", how="left")
gdf = gdf[~gdf.kommune_id.isin(MAP_DROP)].copy()
fig, ax = plt.subplots(figsize=(6, 7))
gdf.plot(ax=ax, color="#E8E8E8", edgecolor="#B8B8B8", linewidth=0.2)
ok = gdf.dropna(subset=["D"])
ok.plot(ax=ax, column="D", cmap="magma_r", edgecolor="#B8B8B8", linewidth=0.2, legend=True)
core_g = gdf[gdf.kommune_id.isin(URBAN_CORE)]
if len(core_g):
    core_g.plot(ax=ax, facecolor="none", edgecolor="k", linewidth=1.2)
ax.set_axis_off()
ax.set_aspect("equal")
fig.savefig(P.FIG_DIR / "fig1_spectral_map.pdf", bbox_inches="tight")
plt.show()


In [ ]:
def eset(g):
    return set(tuple(sorted(e)) for e in g.edges())


edgesets = {k: eset(g) for k, g in graphs.items()}
js = []
for k, e in edgesets.items():
    vals = []
    for u in URBAN_CORE:
        if u == k or u not in edgesets:
            continue
        U = e | edgesets[u]
        vals.append(len(e & edgesets[u]) / len(U) if U else np.nan)
    js.append({"kommune_id": k, "J": np.nanmean(vals)})
jdf = axis.merge(pd.DataFrame(js), on="kommune_id")
print("J to urban core mean", jdf.groupby("group")["J"].mean())
print("J to urban core median", jdf.groupby("group")["J"].median())
print("kruskal J", kruskal(*[jdf.loc[jdf.group == g, "J"].dropna() for g in ("urban", "near", "mid_far") if (jdf.group == g).any()]))
near = jdf.loc[jdf.group == "near", "J"].dropna()
mid = jdf.loc[jdf.group == "mid_far", "J"].dropna()
print("mannwhitney near vs mid_far", mannwhitneyu(near, mid, alternative="two-sided"))


In [ ]:
def Hdeg(g):
    k = np.array([d for _, d in g.degree()])
    n = len(k)
    if n <= 1:
        return np.nan, np.nan
    _, c = np.unique(k, return_counts=True)
    p = c / n
    H = float(-(p * np.log(p)).sum())
    return H, H / np.log(n)


def Hrw(g):
    m = g.number_of_edges()
    n = g.number_of_nodes()
    if m == 0 or n <= 1:
        return np.nan
    k = np.array([d for _, d in g.degree()], dtype=float)
    k = k[k > 0]
    return float((k * np.log(k)).sum() / (2 * m))


cph, nd = URBAN_CORE[0], NORDDJURS
shared = set(graphs[cph].nodes()) & set(graphs[nd].nodes())
print("shared nodes", len(shared))
Gc = graphs[cph].subgraph(shared).copy()
Gn = graphs[nd].subgraph(shared).copy()
print("cph", Hdeg(Gc), Hrw(Gc), "norddjurs", Hdeg(Gn), Hrw(Gn))

ent = []
for k, g in graphs.items():
    hd, hdn = Hdeg(g)
    ent.append({"kommune_id": k, "Hdeg": hd, "Hdeg_n": hdn, "Hrw": Hrw(g)})
ent = axis.merge(pd.DataFrame(ent), on="kommune_id")
print(ent[["D", "Hdeg_n", "Hrw"]].corr(method="spearman"))

union = nx.Graph()
union.add_nodes_from(shared)
union.add_edges_from(Gc.edges())
union.add_edges_from(Gn.edges())
pos = nx.spring_layout(union, seed=42, k=1.2 / np.sqrt(max(len(shared), 1)))
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
for ax, g, title in ((axs[0], Gn, "Norddjurs"), (axs[1], Gc, "København")):
    nx.draw_networkx_edges(g, pos, ax=ax, alpha=0.15, width=0.3, edge_color="#888")
    nx.draw_networkx_nodes(g, pos, ax=ax, node_size=8, node_color="#333", linewidths=0)
    ax.set_title(title)
    ax.axis("off")
fig.savefig(P.FIG_DIR / "fig1_cph_norddjurs.pdf", bbox_inches="tight")
plt.show()


In [ ]:
share_rows = []
for p in files:
    kid = p.stem
    df = pd.read_parquet(p)
    if "weight" not in df.columns:
        df["weight"] = 1.0
    df = df[df.product_i.isin(keep) & df.product_j.isin(keep)]
    tot = float(df.weight.sum())
    acc = defaultdict(float)
    for a, b, w in df[["product_i", "product_j", "weight"]].itertuples(index=False, name=None):
        ca = node_cat.get(str(a), "Other")
        cb = node_cat.get(str(b), "Other")
        pk = pair_key(ca, cb)
        if pk is not None:
            acc[pk] += float(w)
    for pr, w in acc.items():
        share_rows.append({"kommune_id": kid, "c1": pr[0], "c2": pr[1], "share": w / tot if tot else 0.0, "Nsum": w})
shares = pd.DataFrame(share_rows).merge(axis, on="kommune_id")
core_mean = shares[shares.group == "urban"].groupby(["c1", "c2"])["share"].mean()
sel = FADING + RISING + CTRL
for pr in sel:
    sub = shares[(shares.c1 == pr[0]) & (shares.c2 == pr[1])]
    print(pr, spearmanr(sub.D, sub.share, nan_policy="omit"))
ctrl = CTRL[0]
den = core_mean.get(ctrl, np.nan)
sems = []
for gname in ("urban", "near", "mid_far"):
    sub = shares[(shares.group == gname) & (shares.c1 == ctrl[0]) & (shares.c2 == ctrl[1])]
    if len(sub) > 1 and np.isfinite(den) and den != 0:
        sems.append(float((sub.share / den).sem()))
band = max(sems) if sems else np.nan
print("meat-cheese SEM band", band)
fig, ax = plt.subplots(figsize=(8, 4.2))
if np.isfinite(band):
    ax.axhspan(1 - band, 1 + band, color="0.75", alpha=0.45, zorder=0)
order = ["urban", "near", "mid_far"]
x = np.arange(3)
cols_f = ["#0072B2", "#56B4E9", "#009E73", "#66A61E"]
cols_r = ["#882255", "#D55E00", "#CC79A7", "#AA4499"]
for i, pr in enumerate(FADING):
    ys = []
    for g in order:
        sub = shares[(shares.group == g) & (shares.c1 == pr[0]) & (shares.c2 == pr[1])]
        m = sub.share.mean() if len(sub) else np.nan
        den = core_mean.get(pr, np.nan)
        ys.append(np.nan if (den is np.nan or den == 0) else m / den)
    ax.plot(x, ys, marker="o", color=cols_f[i % 4], label="%s–%s" % pr)
for i, pr in enumerate(RISING):
    ys = []
    for g in order:
        sub = shares[(shares.group == g) & (shares.c1 == pr[0]) & (shares.c2 == pr[1])]
        m = sub.share.mean() if len(sub) else np.nan
        den = core_mean.get(pr, np.nan)
        ys.append(np.nan if (den is np.nan or den == 0) else m / den)
    ax.plot(x, ys, marker="s", color=cols_r[i % 4], label="%s–%s" % pr)
ax.axhline(1, color="0.6", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylabel("share / urban core")
ax.legend(fontsize=7, ncol=2, frameon=False)
fig.savefig(P.FIG_DIR / "fig1_meta_pairs.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for pr, mk in zip(sel, "oovvssD^P"):
    yu = shares[(shares.group == "urban") & (shares.c1 == pr[0]) & (shares.c2 == pr[1])].share.mean()
    yf = shares[(shares.group == "mid_far") & (shares.c1 == pr[0]) & (shares.c2 == pr[1])].share.mean()
    ax.plot([100 * yu, 100 * yf], [pr[0] + "–" + pr[1], pr[0] + "–" + pr[1]], color="0.7")
    ax.scatter([100 * yu], [pr[0] + "–" + pr[1]], marker="o", facecolors="none", edgecolors="k")
    ax.scatter([100 * yf], [pr[0] + "–" + pr[1]], marker="s", color="k")
ax.set_xlabel("mean share of N_ij (%)")
fig.savefig(P.FIG_DIR / "fig1_meta_pairs_abs.pdf", bbox_inches="tight")
plt.show()


## fig 2


In [ ]:
def canon(df):
    out = df.copy()
    a = out.product_i.astype(str)
    b = out.product_j.astype(str)
    sw = a > b
    out.loc[sw, "product_i"] = b[sw].values
    out.loc[sw, "product_j"] = a[sw].values
    return out


def lift(nij, ni, nj, n):
    d = ni.astype(float) * nj.astype(float)
    x = nij.astype(float) * n.astype(float) / d
    return x.where(d > 0)


nat = pd.read_parquet(P.NATIONAL_EDGES)
nat = nat[nat.product_i.isin(keep) & nat.product_j.isin(keep)]
nat_z = nat[nat.z_score >= ZSTAR]
Gnat = nx.Graph()
Gnat.add_edges_from(zip(nat_z.product_i.astype(str), nat_z.product_j.astype(str)))
Gnat.remove_nodes_from([n for n, d in Gnat.degree() if d == 0])
lcc = Gnat.subgraph(max(nx.connected_components(Gnat), key=len)).copy()
print("national z*", ZSTAR, lcc.number_of_nodes(), lcc.number_of_edges())

if not P.PARTITION.exists():
    raise FileNotFoundError(
        "paths.PARTITION is missing. Paper community names stay on that file; Louvain is not used to invent ids."
    )
part_df = pd.read_parquet(P.PARTITION)
part = dict(zip(part_df.brand_micro_cat.astype(str), part_df.community_id.astype(int)))
print("loaded partition", len(part))

part = {n: int(part[n]) for n in lcc.nodes() if n in part}
sizes = pd.Series(part).value_counts()
named_ids = [cid for cid, n in sizes.items() if n > MIN_COMM_SIZE and cid in COMM]
node_comm = {n: COMM[cid] for n, cid in part.items() if cid in named_ids}
comm_names = [COMM[cid] for cid in sizes.index if cid in named_ids]
intra_rows = []
for u, v in lcc.edges():
    cu, cv = node_comm.get(u), node_comm.get(v)
    if cu is None or cv is None or cu != cv:
        continue
    a, b = (u, v) if u < v else (v, u)
    intra_rows.append({"product_i": a, "product_j": b, "community_name": cu})
intra = pd.DataFrame(intra_rows).drop_duplicates(["product_i", "product_j"])
print("named", comm_names)
print("intra", len(intra))
Q = None
try:
    import community as community_louvain
    Q = community_louvain.modularity({n: part[n] for n in lcc.nodes() if n in part}, lcc)
    print("Q", Q)
except Exception as e:
    print("Q skip", e)


In [ ]:
stats = pd.read_parquet(P.PAIR_STATS)
stats = canon(stats)
agg_rows = []
for kid, sk in stats.groupby("kommune_id"):
    kid = str(kid)
    if kid in DROP or kid not in graphs:
        continue
    m = sk.merge(intra, on=["product_i", "product_j"], how="inner")
    m = m[m.N_ij >= 1]
    if m.empty:
        continue
    m["lift"] = lift(m.N_ij, m.n_i, m.n_j, m.N)
    m = m[np.isfinite(m.lift) & (m.lift > 0)].copy()
    if m.empty:
        continue
    m["log_lift"] = np.log(m.lift.astype(float))
    m["log_Nij"] = np.log(m.N_ij.astype(float))
    m["log_ni"] = np.log(m.n_i.astype(float))
    m["log_nj"] = np.log(m.n_j.astype(float))
    m["log_M"] = m["log_ni"] + m["log_nj"]
    g = m.groupby("community_name").agg(
        mean_log_lift=("log_lift", "mean"),
        mean_log_Nij=("log_Nij", "mean"),
        mean_log_ni=("log_ni", "mean"),
        mean_log_nj=("log_nj", "mean"),
        mean_log_N=("N", lambda s: float(np.log(float(s.iloc[0])))),
        mean_M=("log_M", "mean"),
        n_edges=("log_lift", "size"),
    ).reset_index()
    g["kommune_id"] = str(kid)
    agg_rows.append(g)
agg = pd.concat(agg_rows, ignore_index=True)
print(agg.head())
print("agg kommuner", agg.kommune_id.nunique())


In [ ]:
def health_join(df_net, spec):
    h = pd.read_csv(P.HEALTH).rename(columns={"municipality": "NAME_2"})
    h["NAME_2"] = h["NAME_2"].replace(NAME_FIX)
    h = h.rename(columns=RATE_MAP)
    for c in ("A10", "C", "C10"):
        if c not in h.columns:
            print("missing", c, h.columns.tolist())
    if spec == "Total":
        controls = [c for c in S_TOTAL if c in h.columns]
        out = df_net.merge(h, on="NAME_2", how="inner")
        return out, controls
    strat = pd.read_csv(P.HEALTH_AGE).rename(columns={"municipality": "NAME_2"})
    strat["NAME_2"] = strat["NAME_2"].replace(NAME_FIX)
    strat = strat.rename(columns=RATE_MAP)
    h60 = strat.loc[strat["age_group"].astype(str) == "60+", ["NAME_2", "A10", "C", "C10"]].drop_duplicates("NAME_2")
    controls = [c for c in S_60 if c in h.columns]
    hs = h[["NAME_2"] + controls].drop_duplicates("NAME_2")
    out = df_net.merge(h60, on="NAME_2").merge(hs, on="NAME_2")
    return out, controls


def pivot_I(agg, names):
    w = agg.pivot_table(index="kommune_id", columns="community_name", values="mean_log_lift", aggfunc="first")
    return w.reindex(columns=names).reset_index()


def kept_comms(agg, df, names):
    n_e = agg.pivot_table(index="kommune_id", columns="community_name", values="n_edges", aggfunc="first").reindex(columns=names).fillna(0)
    kept = []
    ids = df.kommune_id[df.kommune_id.isin(n_e.index)]
    for c in names:
        col = "L__" + c
        if col not in df.columns or df[col].notna().mean() < 0.85:
            continue
        if c in n_e.columns and len(ids) and float(n_e.loc[ids, c].min()) < MIN_EDGES:
            continue
        kept.append(c)
    return kept


def residualize(agg, kept, kids):
    recs = []
    for c in kept:
        sub = agg[(agg.community_name == c) & (agg.kommune_id.isin(kids))].copy()
        sub["P"] = sub.mean_log_Nij - sub.mean_log_N
        sub["Pi"] = sub.mean_log_ni - sub.mean_log_N
        sub["Pj"] = sub.mean_log_nj - sub.mean_log_N
        X = sm.add_constant(sub[["Pi", "Pj"]].astype(float))
        fit = sm.OLS(sub["P"].astype(float), X).fit()
        sub["Pt"] = fit.resid
        recs.append(sub[["kommune_id", "community_name", "Pt"]])
    return pd.concat(recs, ignore_index=True)


def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))


def loo_r2(X, y):
    Xn = np.asarray(X, float)
    yv = np.asarray(y, float)
    pred = cross_val_predict(ridge(), Xn, yv, cv=LeaveOneOut())
    return float(r2_score(yv, pred))


meta = geo[["kommune_id", "NAME_2"]].drop_duplicates("kommune_id") if "NAME_2" in geo.columns else geo[["kommune_id"]].copy()
if "NAME_2" not in meta.columns:
    meta["NAME_2"] = meta["kommune_id"]
lift_w = pivot_I(agg, comm_names)
lift_w = lift_w.rename(columns={c: "L__" + c for c in comm_names if c in lift_w.columns})
df_net = meta.merge(lift_w, on="kommune_id", how="left")
Mw = agg.pivot_table(index="kommune_id", columns="community_name", values="mean_M", aggfunc="first")
Mw = Mw.reindex(columns=comm_names).reset_index()
Mw = Mw.rename(columns={c: "M__" + c for c in comm_names if c in Mw.columns})
df_net = df_net.merge(Mw, on="kommune_id", how="left")

r2tab = []
frames = {}
for spec in ("Total", "60plus"):
    df, controls = health_join(df_net, spec)
    kept = kept_comms(agg, df, comm_names)
    lcols = ["L__" + c for c in kept]
    use = df.dropna(subset=lcols + controls + ["A10", "C", "C10"]).copy()
    use = use[use.kommune_id.isin(graphs)].copy()
    mcols = ["M__" + c for c in kept if "M__" + c in use.columns]
    print(spec, "N", len(use), "I", kept, "S", controls)
    frames[spec] = (use, controls, kept)
    for yname in ("A10", "C", "C10"):
        y = use[yname].astype(float).values
        rI = loo_r2(use[lcols], y)
        rS = loo_r2(use[controls], y)
        rSI = loo_r2(use[controls + lcols], y)
        rSM = loo_r2(use[controls + mcols], y) if mcols else np.nan
        pt = residualize(agg, kept, set(use.kommune_id))
        ptw = pt.pivot_table(index="kommune_id", columns="community_name", values="Pt")
        ptw = ptw.reindex(index=use.kommune_id.values, columns=kept)
        okp = ptw.notna().all(axis=1).values
        rPt = np.nan
        if okp.sum() > 20:
            Xp = np.hstack([use.loc[okp, controls].astype(float).values, ptw.loc[okp].values])
            rPt = loo_r2(Xp, y[okp])
        def resid_share(a, b):
            den = 1.0 - a
            if not np.isfinite(a) or not np.isfinite(b) or den <= 1e-12:
                return np.nan
            return float((b - a) / den)
        r2tab.append({
            "spec": spec, "y": yname, "I": rI, "S": rS, "S_I": rSI, "S_M": rSM, "S_Pt": rPt,
            "SI_resid": resid_share(rS, rSI), "Pt_resid": resid_share(rS, rPt),
        })
        print(spec, yname, "I", round(rI, 3), "S", round(rS, 3), "S+I", round(rSI, 3), "S+M", rSM, "S+Pt", rPt, "SI resid", r2tab[-1]["SI_resid"], "Pt resid", r2tab[-1]["Pt_resid"])
r2tab = pd.DataFrame(r2tab)
r2tab.to_csv(P.FIG_DIR / "fig2_loo_r2.csv", index=False)


In [ ]:
useT, _, _ = frames["Total"]
use60, _, _ = frames["60plus"]
gmap = geo.merge(useT[["kommune_id", "C", "A10"]], on="kommune_id", how="left", suffixes=("", "_r"))
if "C_r" in gmap.columns:
    gmap["C"] = gmap["C_r"]
gmap = gmap[~gmap.kommune_id.isin(MAP_DROP)]
g60 = geo.merge(use60[["kommune_id", "A10"]].rename(columns={"A10": "A10_60"}), on="kommune_id", how="left")
g60 = g60[~g60.kommune_id.isin(MAP_DROP)]
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
gmap.plot(ax=axs[0], color="#E8E8E8", linewidth=0.2, edgecolor="#B8B8B8")
gmap.dropna(subset=["C"]).plot(ax=axs[0], column="C", cmap="Blues", linewidth=0.2, edgecolor="#B8B8B8", legend=True)
axs[0].set_title("C all ages")
axs[0].set_axis_off()
g60.plot(ax=axs[1], color="#E8E8E8", linewidth=0.2, edgecolor="#B8B8B8")
g60.dropna(subset=["A10_60"]).plot(ax=axs[1], column="A10_60", cmap="YlOrRd", linewidth=0.2, edgecolor="#B8B8B8", legend=True)
axs[1].set_title("A10 >=60")
axs[1].set_axis_off()
fig.savefig(P.FIG_DIR / "fig2_health_maps.pdf", bbox_inches="tight")
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
cols = {"I": "#D55E00", "S": "#0072B2", "S_I": "#009E73"}
for ax, spec, models in (
    (axs[0], "Total", ["I", "S", "S_I"]),
    (axs[1], "60plus", ["I", "S_I"]),
):
    sub = r2tab[r2tab.spec == spec]
    x = np.arange(len(sub))
    w = 0.8 / len(models)
    for i, m in enumerate(models):
        ax.bar(x + (i - (len(models) - 1) / 2) * w, sub[m].values, w * 0.9, color=cols[m], label=m.replace("_", "+"))
    ax.set_xticks(x)
    ax.set_xticklabels(sub.y.values)
    ax.set_title(spec)
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, fontsize=8)
axs[0].set_ylabel("LOO R2")
fig.savefig(P.FIG_DIR / "fig2_loo_bars.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5, 3.5))
sub = r2tab[r2tab.spec == "Total"]
x = np.arange(len(sub))
w = 0.35
ax.bar(x - w / 2, sub.S, w, color="#0072B2", label="S")
ax.bar(x + w / 2, sub.S_Pt, w, color="#CC79A7", label="S+Pt")
for i, (sv, pv, resid) in enumerate(zip(sub.S, sub.S_Pt, sub.Pt_resid)):
    if np.isfinite(resid):
        ax.text(i, max(sv, pv) + 0.03, "%d%%" % round(100 * resid), ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(sub.y)
ax.set_ylabel("LOO R2")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8)
fig.savefig(P.FIG_DIR / "fig2_ptilde_bars.pdf", bbox_inches="tight")
plt.show()
print(r2tab[["spec", "y", "S", "S_I", "S_M", "S_Pt", "SI_resid", "Pt_resid"]])


## fig 3


In [ ]:

cph = graphs[URBAN_CORE[0]].copy()
cph.remove_nodes_from(list(nx.isolates(cph)))
e14, eoth = [], []
for u, v in cph.edges():
    pair = {nova_map.get(u), nova_map.get(v)}
    if pair == {1, 4}:
        e14.append((u, v))
    else:
        eoth.append((u, v))
scored = [n for n in cph if n in fs_map]
plain = [n for n in cph if n not in fs_map]
pos = nx.spring_layout(cph, seed=1)
fig, ax = plt.subplots(figsize=(6.2, 6.2))
nx.draw_networkx_edges(cph, pos, edgelist=eoth, ax=ax, edge_color="#B0B0B0", width=0.25, alpha=0.35)
nx.draw_networkx_edges(cph, pos, edgelist=e14, ax=ax, edge_color="#E69F00", width=0.45, alpha=0.7)
if plain:
    nx.draw_networkx_nodes(cph, pos, nodelist=plain, ax=ax, node_size=8, node_color="#BDBDBD", linewidths=0)
if scored:
    vals = [fs_map[n] for n in scored]
    nx.draw_networkx_nodes(
        cph, pos, nodelist=scored, ax=ax, node_size=12,
        node_color=vals, cmap="YlOrRd", vmin=0, vmax=1, linewidths=0,
    )
ax.set_title("København")
ax.axis("off")
fig.savefig(P.FIG_DIR / "fig3_cph_fscore.pdf", bbox_inches="tight")
plt.show()
print("cph nodes", cph.number_of_nodes(), "1-4 edges", len(e14))


In [ ]:
upf_rows = []
rng = np.random.default_rng(0)
for kid, g in graphs.items():
    nodes = [n for n in g.nodes if n in nova_map and n in fs_map]
    H = g.subgraph(nodes).copy()
    H.remove_nodes_from(list(nx.isolates(H)))
    if H.number_of_nodes() < 30:
        continue
    nova = np.array([nova_map[n] for n in H])
    for n in H.nodes:
        H.nodes[n]["fscore"] = fs_map[n]
    rfs = nx.numeric_assortativity_coefficient(H, "fscore")
    rd = nx.degree_assortativity_coefficient(H)
    zs = []
    labs = np.array([fs_map[n] for n in H])
    names = list(H.nodes)
    for _ in range(40):
        rng.shuffle(labs)
        for n, v in zip(names, labs):
            H.nodes[n]["fsh"] = float(v)
        zs.append(nx.numeric_assortativity_coefficient(H, "fsh"))
    z = (rfs - np.mean(zs)) / (np.std(zs) + 1e-12)
    d4 = [H.degree(n) for n, nv in zip(H.nodes, nova) if nv == 4]
    d1 = [H.degree(n) for n, nv in zip(H.nodes, nova) if nv == 1]
    if len(d4) and len(d1):
        U = mannwhitneyu(d4, d1, alternative="two-sided").statistic
        auc = float(U / (len(d4) * len(d1)))
    else:
        auc = np.nan
    p4 = float((nova == 4).mean())
    upf_rows.append({
        "kommune_id": kid, "p4": p4, "r_fscore": rfs, "r_deg": rd,
        "delta_r": rfs - rd, "z_fscore": z, "auc": auc,
    })
upf = pd.DataFrame(upf_rows).merge(axis, on="kommune_id")

a = stats[["kommune_id", "product_i", "n_i"]].rename(columns={"product_i": "p", "n_i": "n"})
b = stats[["kommune_id", "product_j", "n_j"]].rename(columns={"product_j": "p", "n_j": "n"})
nodes = pd.concat([a, b], ignore_index=True)
nodes["kommune_id"] = nodes.kommune_id.astype(str)
nodes = nodes.drop_duplicates(["kommune_id", "p"])
nodes["nova"] = nodes.p.map(nova_map)
nodes = nodes.dropna(subset=["nova", "n"])
nodes = nodes[nodes.n > 0]

nodes["w4"] = (nodes.nova == 4).astype(float) * nodes.n.astype(float)
gn = nodes.groupby("kommune_id")
pupf = (gn["w4"].sum() / gn["n"].sum()).rename("P_UPF").reset_index()
upf = upf.merge(pupf, on="kommune_id", how="left")
print(upf[["D", "p4", "P_UPF", "r_fscore", "auc"]].corr(method="spearman"))
print("auc<0.5", (upf.auc < 0.5).mean(), "mean auc", upf.auc.mean())
print("r_fscore vs D", spearmanr(upf.D, upf.r_fscore, nan_policy="omit"))
print("P_UPF vs D", spearmanr(upf.D, upf.P_UPF, nan_policy="omit"))
print("P_UPF range", upf.P_UPF.min(), upf.P_UPF.max())

crows = []
for kid, sk in stats.groupby("kommune_id"):
    kid = str(kid)
    if kid not in graphs:
        continue
    sk = sk.copy()
    sk["ni"] = sk.product_i.map(nova_map)
    sk["nj"] = sk.product_j.map(nova_map)
    sk = sk.dropna(subset=["ni", "nj"])
    tot = float(sk.N_ij.astype(float).sum())
    if tot <= 0 or sk.empty:
        continue
    N = float(sk.N.iloc[0])
    m44 = sk[(sk.ni == 4) & (sk.nj == 4)]
    C44 = float(m44.N_ij.astype(float).sum() / tot)
    if N > 0 and len(m44):
        dC = float(((m44.N_ij.astype(float) / N) - (m44.n_i.astype(float) * m44.n_j.astype(float) / (N * N))).sum())
    else:
        dC = np.nan
    crows.append({"kommune_id": kid, "C44": C44, "dC44": dC})
upf = upf.merge(pd.DataFrame(crows), on="kommune_id", how="left")
print("C44 vs D", spearmanr(upf.D, upf.C44, nan_policy="omit"))
print("dC44 vs D", spearmanr(upf.D, upf.dC44, nan_policy="omit"))


In [ ]:
natc = canon(nat_z)
natc["n4i"] = natc.product_i.map(nova_map)
natc["n4j"] = natc.product_j.map(nova_map)
e44 = natc[(natc.n4i == 4) & (natc.n4j == 4)][["product_i", "product_j"]]
e11 = natc[(natc.n4i == 1) & (natc.n4j == 1)][["product_i", "product_j"]]
print("nat 44", len(e44), "11", len(e11))
ll_rows = []
for kid, sk in stats.groupby("kommune_id"):
    kid = str(kid)
    if kid in DROP:
        continue
    sk = canon(sk)
    m44 = sk.merge(e44, on=["product_i", "product_j"])
    m11 = sk.merge(e11, on=["product_i", "product_j"])
    rec = {"kommune_id": kid}
    for name, m in (("ll44", m44), ("ll11", m11)):
        if m.empty:
            rec[name] = np.nan
            continue
        lv = lift(m.N_ij, m.n_i, m.n_j, m.N)
        lv = lv[np.isfinite(lv) & (lv > 0)]
        rec[name] = float(np.log(lv).mean()) if len(lv) else np.nan
    ll_rows.append(rec)
ll = pd.DataFrame(ll_rows).merge(upf, on="kommune_id")
ll["gap"] = ll.ll44 - ll.ll11
print("44>11", (ll.ll44 > ll.ll11).mean())
print("ll44 vs D", spearmanr(ll.D, ll.ll44, nan_policy="omit"))
print("ll11 vs D", spearmanr(ll.D, ll.ll11, nan_policy="omit"))
print("gap vs D", spearmanr(ll.D, ll.gap, nan_policy="omit"))

fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
axs[0].scatter(ll.D, ll.auc, s=18, c="#444")
axs[0].axhline(0.5, color="0.6", lw=0.7)
axs[0].set_ylabel("AUC 4 vs 1")
axs[1].scatter(ll.D, ll.r_fscore, s=18, c="#444")
axs[1].set_ylabel(r"$r_{FScore}$")
dlt = (ll.r_fscore - ll.r_deg).to_numpy(float)
ax2 = axs[1].twinx()
lo, hi = np.nanpercentile(dlt, [25, 75])
ax2.axhspan(lo, hi, color="0.6", alpha=0.35, zorder=0)
ax2.axhline(np.nanmean(dlt), color="0.25", lw=1.0, ls="--")
ax2.set_ylabel(r"$r_{FScore}-r_{degree}$")
axs[1].set_zorder(ax2.get_zorder() + 1)
axs[1].patch.set_visible(False)
axs[2].scatter(ll.D, ll.ll44, s=18, c="#444", label="4-4")
axs[2].scatter(ll.D, ll.ll11, s=14, facecolors="none", edgecolors="#444", marker="s", label="1-1")
axs[2].set_ylabel("mean log-lift")
axs[2].legend(frameon=False, fontsize=7)
for ax in axs:
    ax.set_xlabel("D")
fig.tight_layout()
fig.savefig(P.FIG_DIR / "fig3_upf.pdf", bbox_inches="tight")
plt.show()

dec_rows = []
for kid, g in graphs.items():
    nodes = [n for n in g.nodes if n in nova_map]
    H = g.subgraph(nodes).copy()
    H.remove_nodes_from(list(nx.isolates(H)))
    if H.number_of_nodes() < 30:
        continue
    deg = pd.Series({n: H.degree(n) for n in H})
    nova = pd.Series({n: nova_map[n] for n in H})
    b = pd.qcut(deg.rank(method="first"), 10, labels=False, duplicates="drop")
    p4loc = float((nova == 4).mean())
    tmp = pd.DataFrame({"bin": b, "nova": nova})
    for bb, sub in tmp.groupby("bin"):
        dec_rows.append({"kommune_id": kid, "bin": int(bb), "p4rel": float((sub.nova == 4).mean()) - p4loc})
dec = pd.DataFrame(dec_rows)
fig, ax = plt.subplots(figsize=(5, 3.5))
for kid, sub in dec.groupby("kommune_id"):
    ax.plot(sub.bin + 1, sub.p4rel, color="0.8", lw=0.6)
mu = dec.groupby("bin").p4rel.mean()
ax.plot(mu.index + 1, mu.values, color="#B40426", lw=2)
ax.axhline(0, color="0.5", lw=0.7)
ax.set_xlabel("degree decile")
ax.set_ylabel("UPF share minus local baseline")
fig.savefig(P.FIG_DIR / "fig3_deciles.pdf", bbox_inches="tight")
plt.show()


## SI


In [ ]:
if "level_2_meso" in prod.columns:
    meso = dict(zip(prod.brand_micro_cat.astype(str), prod.level_2_meso.astype(str)))
    rows = []
    for n, cname in node_comm.items():
        rows.append({"community": SHORT.get(cname, cname), "meso": meso.get(n, "Other")})
    cdf = pd.DataFrame(rows)
    fig, axs = plt.subplots(3, 4, figsize=(12, 8))
    axs = axs.ravel()
    labs = [SHORT.get(c, c) for c in comm_names]
    for i, lab in enumerate(labs[:12]):
        sub = cdf[cdf.community == lab]["meso"].value_counts(normalize=True)
        top = sub.head(6)
        rest = 1 - top.sum()
        ys = list(top.values) + ([rest] if rest > 0.01 else [])
        xs = list(top.index) + (["Other"] if rest > 0.01 else [])
        axs[i].barh(xs[::-1], np.array(ys[::-1]) * 100)
        axs[i].set_title(lab, fontsize=8)
    for j in range(i + 1, len(axs)):
        axs[j].axis("off")
    fig.tight_layout()
    fig.savefig(P.FIG_DIR / "si_community_composition.pdf", bbox_inches="tight")
    plt.show()
else:
    print("no level_2_meso")


In [ ]:
forest = []
for spec in ("Total", "60plus"):
    use, controls, kept = frames[spec]
    pt = residualize(agg, kept, set(use.kommune_id))
    ptw = pt.pivot_table(index="kommune_id", columns="community_name", values="Pt")
    use2 = use.merge(ptw.reset_index(), on="kommune_id", how="left")
    for yname in ("A10", "C", "C10"):
        y = use2[yname].astype(float)
        ysd = float(y.std(ddof=0))
        for c in kept:
            for feat, col in (("I", "L__" + c), ("Pt", c)):
                if col not in use2.columns:
                    continue
                sub = use2[controls + [col]].copy()
                sub["y"] = y
                sub = sub.dropna()
                Xs = StandardScaler().fit_transform(sub[controls + [col]].astype(float))
                Xs = sm.add_constant(Xs, has_constant="add")
                fit = sm.OLS(sub["y"].values, Xs).fit(cov_type="HC3")
                idx = 1 + len(controls)
                ci = fit.conf_int()
                forest.append({
                    "spec": spec, "y": yname, "comm": SHORT.get(c, c), "feat": feat,
                    "beta": float(fit.params[idx]) / ysd,
                    "lo": float(ci[idx, 0]) / ysd,
                    "hi": float(ci[idx, 1]) / ysd,
                    "p": float(fit.pvalues[idx]),
                })
forest = pd.DataFrame(forest)
forest.to_csv(P.FIG_DIR / "si_forest.csv", index=False)
fig, axs = plt.subplots(2, 3, figsize=(11, 8), sharex=True)
for r, spec in enumerate(("Total", "60plus")):
    for c, yname in enumerate(("A10", "C", "C10")):
        ax = axs[r, c]
        sub = forest[(forest.spec == spec) & (forest.y == yname)]
        labs = list(dict.fromkeys(sub.comm))
        for feat, col, off, mk in (("I", "#0072B2", -0.15, "o"), ("Pt", "#E69F00", 0.15, "s")):
            ss = sub[sub.feat == feat].set_index("comm").reindex(labs)
            y = np.arange(len(labs)) + off
            xerr = np.vstack([ss.beta - ss.lo, ss.hi - ss.beta])
            sig = (ss.p < 0.05).fillna(False).to_numpy()
            labeled = False
            for mask, mfc in ((sig, col), (~sig, "none")):
                if not mask.any():
                    continue
                ax.errorbar(
                    ss.beta.to_numpy()[mask], y[mask], xerr=xerr[:, mask],
                    fmt=mk, color=col, ms=4, mfc=mfc, linestyle="none",
                    label=feat if not labeled else None,
                )
                labeled = True
        ax.axvline(0, color="0.6", lw=0.7)
        ax.set_yticks(np.arange(len(labs)))
        ax.set_yticklabels(labs, fontsize=7)
        ax.set_title("%s %s" % (spec, yname), fontsize=9)
axs[0, 0].legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(P.FIG_DIR / "si_forest.pdf", bbox_inches="tight")
plt.show()


In [ ]:
from libpysal.weights import KNN
from esda.moran import Moran

use, controls, kept = frames["Total"]
lcols = ["L__" + c for c in kept]
guse = geo.merge(use.drop(columns=["geometry"], errors="ignore"), on="kommune_id", how="inner")
guse = gpd.GeoDataFrame(guse, geometry="geometry", crs=geo.crs).reset_index(drop=True)
cent = np.column_stack([guse.geometry.centroid.x, guse.geometry.centroid.y])
w = KNN.from_array(cent, k=5)
w.transform = "r"


def ols_resid(X, y):
    yv = np.asarray(y, float)
    if X is None:
        return yv - yv.mean()
    Xs = sm.add_constant(StandardScaler().fit_transform(np.asarray(X, float)), has_constant="add")
    return np.asarray(sm.OLS(yv, Xs).fit().resid)


def neigh_r2(X, y, w):
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    pred = np.full(len(y), np.nan)
    for i in range(len(y)):
        hold = {i} | set(int(j) for j in w.neighbors.get(i, []))
        tr = np.array([j for j in range(len(y)) if j not in hold])
        if len(tr) < X.shape[1] + 2:
            continue
        est = ridge()
        est.fit(X[tr], y[tr])
        pred[i] = est.predict(X[[i]])[0]
    ok = np.isfinite(pred)
    return float(r2_score(y[ok], pred[ok]))


def region_r2(X, y, groups):
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    pred = np.full(len(y), np.nan)
    for g in pd.unique(groups):
        te = groups == g
        tr = ~te
        if tr.sum() < X.shape[1] + 2:
            continue
        est = ridge()
        est.fit(X[tr], y[tr])
        pred[te] = est.predict(X[te])
    ok = np.isfinite(pred)
    return float(r2_score(y[ok], pred[ok]))


moran_rows = []
cv_rows = []
for spec in ("Total", "60plus"):
    use, controls, kept = frames[spec]
    lcols = ["L__" + c for c in kept]
    guse = geo.merge(use.drop(columns=["geometry"], errors="ignore"), on="kommune_id", how="inner")
    guse = gpd.GeoDataFrame(guse, geometry="geometry", crs=geo.crs).reset_index(drop=True)
    cent = np.column_stack([guse.geometry.centroid.x.values, guse.geometry.centroid.y.values])
    w = KNN.from_array(cent, k=5)
    w.transform = "r"
    groups = guse["NAME_1"].astype(str).values if "NAME_1" in guse.columns else np.array(["x"] * len(guse))
    for yname in ("A10", "C", "C10"):
        y = guse[yname].astype(float).values
        for lab, X in (("raw", None), ("S", guse[controls]), ("S+I", guse[controls + lcols])):
            resid = ols_resid(X, y)
            mi = Moran(resid, w, permutations=999)
            moran_rows.append({"spec": spec, "y": yname, "model": lab, "I": float(mi.I), "p": float(mi.p_sim)})
        cv_rows.append({
            "spec": spec, "y": yname,
            "loo_S": loo_r2(guse[controls], y),
            "loo_SI": loo_r2(guse[controls + lcols], y),
            "nb_S": neigh_r2(guse[controls], y, w),
            "nb_SI": neigh_r2(guse[controls + lcols], y, w),
            "reg_S": region_r2(guse[controls], y, groups),
            "reg_SI": region_r2(guse[controls + lcols], y, groups),
        })
moran = pd.DataFrame(moran_rows)
cv = pd.DataFrame(cv_rows)
print(moran)
print(cv)
moran.to_csv(P.FIG_DIR / "si_moran.csv", index=False)
cv.to_csv(P.FIG_DIR / "si_spatial_cv.csv", index=False)

fig, axs = plt.subplots(2, 3, figsize=(10, 6), sharey=True)
for r, spec in enumerate(("Total", "60plus")):
    for c, yname in enumerate(("A10", "C", "C10")):
        ax = axs[r, c]
        row = cv[(cv.spec == spec) & (cv.y == yname)].iloc[0]
        labs = ["loo", "neigh", "region"]
        S = [row.loo_S, row.nb_S, row.reg_S]
        SI = [row.loo_SI, row.nb_SI, row.reg_SI]
        y = np.arange(3)
        ax.hlines(y - 0.12, 0, np.clip(S, 0, None), color="#0072B2")
        ax.plot(np.clip(S, 0, None), y - 0.12, "o", color="#0072B2")
        ax.hlines(y + 0.12, 0, np.clip(SI, 0, None), color="#009E73")
        ax.plot(np.clip(SI, 0, None), y + 0.12, "o", color="#009E73")
        ax.set_yticks(y)
        ax.set_yticklabels(labs)
        ax.set_title("%s %s" % (spec, yname), fontsize=9)
        ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(P.FIG_DIR / "si_spatial_cv.pdf", bbox_inches="tight")
plt.show()


In [ ]:
zs = np.arange(5, 46, 2.5)
sweep = []
base = canon(nat)
for z in zs:
    e = base[base.z_score >= z]
    g = nx.Graph()
    g.add_edges_from(zip(e.product_i.astype(str), e.product_j.astype(str)))
    g.remove_nodes_from([n for n, d in g.degree() if d == 0])
    n = g.number_of_nodes()
    if n == 0:
        sweep.append({"z": z, "edges": 0, "lcc": 0, "Q": np.nan})
        continue
    cc = max(nx.connected_components(g), key=len)
    h = g.subgraph(cc).copy()
    try:
        import community as community_louvain
        pr = community_louvain.best_partition(h, resolution=1.0, random_state=42)
        Q = community_louvain.modularity(pr, h)
    except Exception:
        Q = np.nan
    sweep.append({"z": float(z), "edges": g.number_of_edges(), "lcc": len(cc) / n, "Q": Q})
    print(z, sweep[-1])
sweep = pd.DataFrame(sweep)
sweep.to_csv(P.FIG_DIR / "si_zscore_sweep.csv", index=False)
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
axs[0].plot(sweep.z, sweep.edges / 1000.0)
axs[0].axvline(ZSTAR, color="0.5", ls="--")
axs[0].set_ylabel("edges (k)")
axs[1].plot(sweep.z, sweep.lcc)
axs[1].axhline(0.95, color="0.5", ls="--")
axs[1].axvline(ZSTAR, color="0.5", ls="--")
axs[1].set_ylabel("LCC frac")
axs[2].plot(sweep.z, sweep.Q)
axs[2].axvline(ZSTAR, color="0.5", ls="--")
axs[2].set_ylabel("Q")
for ax in axs:
    ax.set_xlabel("z")
fig.tight_layout()
fig.savefig(P.FIG_DIR / "si_zscore_sweep.pdf", bbox_inches="tight")
plt.show()


In [ ]:
if P.TIE_CARRIAGE.exists():
    tc = pd.read_parquet(P.TIE_CARRIAGE) if str(P.TIE_CARRIAGE).endswith("parquet") else pd.read_csv(P.TIE_CARRIAGE)
    print(tc.head())
    ycol = "gamma" if "gamma" in tc.columns else tc.columns[-1]
    cls = "pair_class" if "pair_class" in tc.columns else tc.columns[0]
    fig, ax = plt.subplots(figsize=(5, 3.5))
    data = []
    labs = []
    for k in ("11", "1-1", "14", "1-4", "44", "4-4"):
        if k in set(tc[cls].astype(str)):
            data.append(tc.loc[tc[cls].astype(str) == k, ycol].dropna().values)
            labs.append(k)
    ax.violinplot(data, showmeans=True)
    ax.set_xticks(np.arange(1, len(labs) + 1))
    ax.set_xticklabels(labs)
    ax.set_ylabel(ycol)
    fig.savefig(P.FIG_DIR / "si_tie_carriage.pdf", bbox_inches="tight")
    plt.show()
else:
    print("no tie carriage file, skip")
